# Word LLM Translation Workflow — Evaluation Stage

- **Workflow stage:** `evaluation_completed` (written at the end of this notebook)
- **Input checkpoint:** primary translation checkpoint from the previous stage
- **Source document:** loaded from checkpoint metadata
- **Source language:** loaded from checkpoint metadata
- **Target language:** loaded from checkpoint metadata
- **Primary translator model:** loaded from checkpoint metadata
- **Evaluator model:** user-defined in this notebook
- **Purpose of this notebook:** evaluate the primary translation output element by element using batched OpenAI requests

## Purpose of this notebook
This notebook evaluates the primary translation output produced in the earlier stage. It reuses the saved source language, target language, primary model provenance, and primary translation results from the checkpoint, then applies an evaluator model to determine whether each translated element passes or fails.

## Evaluator model note

In this workflow, the evaluator stage is configured to use an OpenAI/GPT model. Do not substitute a different provider or model API path here unless you also update the corresponding evaluator helper functions in `workflow_helpers.py`, since the current implementation is written for the OpenAI client and response format.

## Metadata flow
This notebook:

- loads `metadata` and `elements` from the prior checkpoint
- pulls inherited workflow settings from checkpoint metadata
- defines and records the evaluator model name
- defines and records the evaluator system message
- writes evaluator results back into the schema fields
- saves an updated checkpoint with both `metadata` and `elements`

## Notes
- Evaluation is performed in batches using the existing `batch_number` groupings.
- Pass/fail decisions are returned per element, not per batch.
- This notebook does not perform fallback translation.
- Fallback translation is handled in a later notebook only for elements that fail evaluation.

### Setup

In [1]:
# User-defined models for this notebook stage
openai_model = "gpt-5.4-mini"

In [2]:
# show json files in the /checkpoints subdirectory
import importlib
import workflow_helpers
workflow_helpers = importlib.reload(workflow_helpers)

checkpoints_dir = workflow_helpers.get_checkpoints_dir()
json_files = workflow_helpers.list_json_files(checkpoints_dir)

Found JSON files:
- elements_batched_20260411_1519.json
- primary_retry_completed_1_20260411_1541.json
- primary_translation_completed_20260411_1527.json


In [3]:
# Load checkpoint state (metadata + elements)
# filename should contain either primary_translation_completed or primary_retry_completed

import os
from importlib import reload
import workflow_helpers
from pprint import pprint

workflow_helpers = reload(workflow_helpers)

# Define the checkpoint file
checkpoint_dir = "checkpoints"
json_file = "primary_retry_completed_1_20260411_1541.json"

# Full path
checkpoint_path = os.path.join(checkpoint_dir, json_file)

# Load normalized metadata and elements
metadata, elements = workflow_helpers.load_elements_checkpoint(checkpoint_path)

print("Loaded checkpoint:", checkpoint_path)
print("Top-level state loaded through workflow_helpers.load_elements_checkpoint")
print("\nMetadata:")
pprint(metadata)

print(f"\nLoaded checkpoint with {len(elements)} elements.")
print("\nExample entry:")
pprint(elements[0] if elements else None)

Loaded checkpoint: checkpoints\primary_retry_completed_1_20260411_1541.json
Top-level state loaded through workflow_helpers.load_elements_checkpoint

Metadata:
{'docxfilename': 'custom_word_styles_example.docx',
 'evaluation_model_name': None,
 'evaluation_system_message': None,
 'fallback_model_name': None,
 'fallback_system_message': None,
 'primary_model_name': 'gemini-3.1-pro-preview',
 'primary_system_message': 'You are a professional translator of Biblical '
                           'education materials for a Protestant/Evangelical '
                           'audience.\n'
                           '\n'
                           'You will receive a single JSON object with the '
                           'following structure:\n'
                           '\n'
                           '{\n'
                           '  "elements": [\n'
                           '    {\n'
                           '      "id": "<string>",\n'
                           '      "text": "<ch

In [4]:
# Pull inherited workflow settings from checkpoint metadata
source_language = metadata.get("source_language")
target_language = metadata.get("target_language")
gemini_model = metadata.get("primary_model_name")

missing = [
    name for name, value in {
        "source_language": source_language,
        "target_language": target_language,
        "gemini_model": gemini_model,
    }.items()
    if not value
]

if missing:
    raise ValueError(
        f"Missing required metadata field(s) in checkpoint: {', '.join(missing)}"
    )

# User-defined model for this notebook stage
openai_model = "gpt-5.4-mini"

# Record stage-specific metadata
metadata["evaluation_model_name"] = openai_model

print("Source language:", source_language)
print("Target language:", target_language)
print("Primary model:", gemini_model)
print("Evaluation model:", openai_model)

Source language: English
Target language: Traditional Chinese
Primary model: gemini-3.1-pro-preview
Evaluation model: gpt-5.4-mini


### Define Evaluator System Message
#### Adapting the evaluator prompt

This prompt is written for the current example workflow and evaluation criteria, but it is intended to be revised for your own material and target language.

Best practices when adapting it:

- keep the evaluator focused on clear meaning preservation and required formatting integrity rather than stylistic preference
- update domain-specific references so they match your content type (for example, Biblical, legal, technical, educational, or literary material)
- preserve the JSON input/output contract unless you also update the downstream helper code that parses evaluator results
- keep the pass/fail criteria concrete and stable so the evaluator does not become overly strict or inconsistent across items
- avoid overfitting the prompt to a single difficult sentence or isolated edge case
- if you use an LLM to help rewrite the evaluator prompt, review the result carefully to make sure it still matches your intended validation policy
- after changing the prompt, run a one-batch smoke test before launching the full evaluation pass

In general, it is fine to revise the domain guidance and evaluation standards, but be cautious about changing the response schema, field names, or strict JSON-only output rules unless you are also updating the downstream workflow logic.

In [5]:
# Evaluator system prompt

evaluator_system_message = f"""
You are a translation validator for Biblical education materials intended for a Protestant/Evangelical audience.

Your task is to evaluate whether each candidate translation faithfully preserves the meaning of the source text and preserves required Markdown / inline formatting.

Be careful but not overly strict. Allow natural translation variation. Do not fail a translation simply because it is not literal.

FAIL only when there is a clear substantive problem.

CRITICAL PASS/FAIL CHECKS
1) Meaning Preservation
   - The candidate translation must preserve the meaning of the source text without substantive omission, addition, or distortion.
   - Natural reordering for fluency is acceptable.
   - Do not fail for minor differences in wording if the meaning is still clearly preserved.

2) Formatting Integrity
   - Required Markdown or inline formatting must be preserved when present in the source.
   - This includes headings, list markers, blockquotes, backticks, bold, italics, and link syntax where applicable.
   - Minor whitespace differences around formatting markers are acceptable.
   - Ordered lists may use normalized numbering if the structure is preserved.

3) Non-Translatable Content
   - Inline code, URLs, and other clearly non-translatable literal content must remain unchanged unless the source itself changes them.

4) No Extraneous Content
   - The candidate translation must not add commentary, explanations, notes, or boilerplate not present in the source.

TOLERANCES
- Reasonable linguistic variation is acceptable.
- Natural differences in phrasing, word order, punctuation, quotation marks, dashes, or locale-specific orthography are acceptable if meaning is preserved.
- Standard target-language renderings of Biblical names and theological terms are acceptable.
- Multiple acceptable renderings may exist for Biblical and theological wording.
- Do not fail solely because a different wording might be more elegant, more idiomatic, or more conventional.
- Do not fail solely for terminology preference if the candidate still preserves the source meaning in context.
- Evaluate each item independently based only on the provided source and candidate text.

WHEN TO FAIL
Fail only for concrete issues such as:
- missing or added meaning
- clearly mistranslated theological or Biblical content
- broken or missing required formatting
- changed or corrupted non-translatable literal content
- extraneous model-added text

WHEN NOT TO FAIL
Do not fail for:
- non-literal but faithful translation
- acceptable terminology variation
- awkward but still semantically accurate phrasing
- a different but still valid rendering of Biblical or theological wording
- cases where the original error has been corrected and the remaining difference is only a debatable wording preference

FEEDBACK RULES
- If the item passes, set feedback to an empty string.
- If the item fails, give a short, specific reason tied to a clear meaning or formatting problem.
- Do not suggest failure merely because another translation might sound better.

OUTPUT FORMAT
Return ONLY one valid JSON object with this structure:

{{
  "elements": [
    {{
      "element_id": "<copy from input>",
      "passed": true,
      "feedback": ""
    }},
    {{
      "element_id": "<copy from input>",
      "passed": false,
      "feedback": "<brief concrete reason for failure>"
    }}
  ]
}}

INPUT FORMAT
You will receive one JSON object with this structure:

{{
  "elements": [
    {{
      "element_id": "<string>",
      "source_text": "<source text in {source_language}>",
      "candidate_text": "<candidate translation in {target_language}>"
    }}
  ]
}}

OUTPUT RULES
- Return JSON only.
- Do not wrap the JSON in code fences.
- Do not add commentary outside the JSON.
- Preserve input order.
- Include exactly one output object for each input element.
- If an item passes, set "feedback" to an empty string.
- If an item fails, give a short, specific reason.
""".strip()

metadata["evaluation_system_message"] = evaluator_system_message

print("Evaluator system prompt defined and stored in metadata.")
print("Prompt preview:")
print(evaluator_system_message[:500])

Evaluator system prompt defined and stored in metadata.
Prompt preview:
You are a translation validator for Biblical education materials intended for a Protestant/Evangelical audience.

Your task is to evaluate whether each candidate translation faithfully preserves the meaning of the source text and preserves required Markdown / inline formatting.

Be careful but not overly strict. Allow natural translation variation. Do not fail a translation simply because it is not literal.

FAIL only when there is a clear substantive problem.

CRITICAL PASS/FAIL CHECKS
1) Meani


In [6]:
# Identify elements eligible for evaluation

eligible_elements = [
    el for el in elements
    if el.get("primary_translation") is not None
    and el.get("primary_error") is None
    and el.get("final") is None
]

print("Total elements:", len(elements))
print("Eligible for evaluation:", len(eligible_elements))

if eligible_elements:
    print("\nExample eligible element:")
    print({
        "element_id": eligible_elements[0].get("element_id"),
        "batch_number": eligible_elements[0].get("batch_number"),
        "text": eligible_elements[0].get("text"),
        "primary_translation": eligible_elements[0].get("primary_translation"),
    })

Total elements: 29
Eligible for evaluation: 29

Example eligible element:
{'element_id': '8633e724fc99', 'batch_number': 1, 'text': 'Introduction', 'primary_translation': '簡介'}


In [7]:
# Group eligible elements by existing batch_number for evaluator batching

evaluation_batches = {}
for el in eligible_elements:
    batch_number = el.get("batch_number")
    evaluation_batches.setdefault(batch_number, []).append(el)

evaluation_batch_numbers = sorted(evaluation_batches.keys())

print("Evaluation batches detected:", len(evaluation_batch_numbers))
print("First few batch numbers:", evaluation_batch_numbers[:10])

print("\nExample batch sizes:")
for batch_number in evaluation_batch_numbers[:5]:
    print(f"  batch {batch_number}: {len(evaluation_batches[batch_number])} elements")

Evaluation batches detected: 12
First few batch numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Example batch sizes:
  batch 1: 3 elements
  batch 2: 1 elements
  batch 3: 4 elements
  batch 4: 3 elements
  batch 5: 3 elements


In [8]:
# Preview the evaluator payload shape for one batch

preview_batch_number = evaluation_batch_numbers[0]

preview_payload = {
    "elements": [
        {
            "element_id": el["element_id"],
            "source_text": el["text"],
            "candidate_text": el["primary_translation"],
        }
        for el in evaluation_batches[preview_batch_number]
    ]
}

print("Preview batch number:", preview_batch_number)
print("Payload element count:", len(preview_payload["elements"]))
print("\nFirst payload item:")
print(preview_payload["elements"][0])

Preview batch number: 1
Payload element count: 3

First payload item:
{'element_id': '8633e724fc99', 'source_text': 'Introduction', 'candidate_text': '簡介'}


In [9]:
# Smoke test: evaluate a single batch without mutating the full `elements` list

from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

test_elements = [el.copy() for el in elements if el.get("batch_number") == 1]

test_elements = workflow_helpers.run_evaluation_pass(
    elements=test_elements,
    openai_model_name=openai_model,
    evaluator_system_message=evaluator_system_message,
    verbose=True,
    print_status_every_n_batches=1,
)

print("Example evaluated element:")
print({
    "element_id": test_elements[0].get("element_id"),
    "primary_translation": test_elements[0].get("primary_translation"),
    "evaluator_ran": test_elements[0].get("evaluator_ran"),
    "evaluator_passed": test_elements[0].get("evaluator_passed"),
    "evaluator_feedback": test_elements[0].get("evaluator_feedback"),
    "evaluator_error": test_elements[0].get("evaluator_error"),
})

[16:03:45] Starting evaluation: 1 batches detected.
[16:03:48] Progress: 1/1 batches completed.
[16:03:48] Evaluation complete: 1/1 batches processed in 00:00.
Example evaluated element:
{'element_id': '8633e724fc99', 'primary_translation': '簡介', 'evaluator_ran': True, 'evaluator_passed': True, 'evaluator_feedback': '', 'evaluator_error': None}


In [10]:
# Quick QA on the one-batch smoke test

evaluated_count = sum(el.get("evaluator_ran") is True for el in test_elements)
passed_count = sum(el.get("evaluator_passed") is True for el in test_elements)
failed_count = sum(el.get("evaluator_passed") is False for el in test_elements)
error_count = sum(el.get("evaluator_error") is not None for el in test_elements)

print("Evaluated:", evaluated_count)
print("Passed:", passed_count)
print("Failed:", failed_count)
print("Errors:", error_count)

print("\nDetailed results:")
for el in test_elements:
    print({
        "element_id": el.get("element_id"),
        "evaluator_passed": el.get("evaluator_passed"),
        "evaluator_feedback": el.get("evaluator_feedback"),
        "evaluator_error": el.get("evaluator_error"),
    })

Evaluated: 3
Passed: 3
Failed: 0
Errors: 0

Detailed results:
{'element_id': '8633e724fc99', 'evaluator_passed': True, 'evaluator_feedback': '', 'evaluator_error': None}
{'element_id': '2e6212d6aad5', 'evaluator_passed': True, 'evaluator_feedback': '', 'evaluator_error': None}
{'element_id': '4126aaaf767e', 'evaluator_passed': True, 'evaluator_feedback': '', 'evaluator_error': None}


### Run all batches

In [11]:
# Run evaluation across all eligible batches

from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

elements = workflow_helpers.run_evaluation_pass(
    elements=elements,
    openai_model_name=openai_model,
    evaluator_system_message=evaluator_system_message,
    verbose=True,
    print_status_every_n_batches=1,
)

[16:03:48] Starting evaluation: 12 batches detected.
[16:03:50] Progress: 1/12 batches completed.
[16:03:51] Progress: 2/12 batches completed.
[16:03:52] Progress: 3/12 batches completed.
[16:03:53] Progress: 4/12 batches completed.
[16:03:54] Progress: 5/12 batches completed.
[16:03:55] Progress: 6/12 batches completed.
[16:03:59] Progress: 7/12 batches completed.
[16:04:00] Progress: 8/12 batches completed.
[16:04:01] Progress: 9/12 batches completed.
[16:04:02] Progress: 10/12 batches completed.
[16:04:03] Progress: 11/12 batches completed.
[16:04:04] Progress: 12/12 batches completed.
[16:04:04] Evaluation complete: 12/12 batches processed in 00:00.


In [12]:
# Quick QA after full evaluation pass

total_elements = len(elements)
evaluated_count = sum(el.get("evaluator_ran") is True for el in elements)
passed_count = sum(el.get("evaluator_passed") is True for el in elements)
failed_count = sum(el.get("evaluator_passed") is False for el in elements)
error_count = sum(el.get("evaluator_error") is not None for el in elements)

print("Total elements:", total_elements)
print("Evaluated:", evaluated_count)
print("Passed:", passed_count)
print("Failed:", failed_count)
print("Errors:", error_count)

if failed_count:
    print("\nExample failed item:")
    for el in elements:
        if el.get("evaluator_passed") is False:
            print({
                "element_id": el.get("element_id"),
                "text": el.get("text"),
                "primary_translation": el.get("primary_translation"),
                "evaluator_feedback": el.get("evaluator_feedback"),
            })
            break

if error_count:
    print("\nExample evaluator error:")
    for el in elements:
        if el.get("evaluator_error") is not None:
            print({
                "element_id": el.get("element_id"),
                "batch_number": el.get("batch_number"),
                "evaluator_error": el.get("evaluator_error"),
            })
            break

Total elements: 29
Evaluated: 29
Passed: 28
Failed: 1
Errors: 0

Example failed item:
{'element_id': '44b22fddd655', 'text': 'The Bible consists of the **Hebrew** and **Greek** Scriptures.', 'primary_translation': '聖經是由**希伯來文**和**希臘文**的經卷所組成。', 'evaluator_feedback': 'The source says Hebrew and Greek Scriptures, but the translation says Hebrew and Greek languages/versions of scrolls, which changes the meaning.'}


In [13]:
# Inspect all failed evaluator items in detail

failed_elements = [
    el for el in elements
    if el.get("evaluator_passed") is False
]

print("Failed count:", len(failed_elements))

for i, el in enumerate(failed_elements, start=1):
    print(f"\n--- Failed item {i} ---")
    print("element_id:", el.get("element_id"))
    print("batch_number:", el.get("batch_number"))
    print("word_style:", el.get("word_style"))
    print("text:")
    print(el.get("text"))
    print("\nprimary_translation:")
    print(el.get("primary_translation"))
    print("\nevaluator_feedback:")
    print(el.get("evaluator_feedback"))
    print("\nevaluator_error:")
    print(el.get("evaluator_error"))

Failed count: 1

--- Failed item 1 ---
element_id: 44b22fddd655
batch_number: 7
word_style: body
text:
The Bible consists of the **Hebrew** and **Greek** Scriptures.

primary_translation:
聖經是由**希伯來文**和**希臘文**的經卷所組成。

evaluator_feedback:
The source says Hebrew and Greek Scriptures, but the translation says Hebrew and Greek languages/versions of scrolls, which changes the meaning.

evaluator_error:
None


In [14]:
metadata["stage"] = workflow_helpers.WORKFLOW_STAGES["evaluation_completed"]

In [15]:
# Assign final output for primary translations that passed evaluation

for el in elements:
    if el.get("evaluator_passed") is True:
        el["final"] = el.get("primary_translation")
        el["final_model"] = el.get("primary_translation_model")

In [16]:
# Reality check after assigning final for passed primary translations

total_elements = len(elements)
final_present = sum(bool((el.get("final") or "").strip()) for el in elements)
final_missing = total_elements - final_present

print("Total elements:", total_elements)
print("Elements with final translation:", final_present)
print("Elements missing final translation:", final_missing)

Total elements: 29
Elements with final translation: 28
Elements missing final translation: 1


In [17]:
# Save evaluation-completed checkpoint

from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

checkpoint_path = workflow_helpers.save_elements_checkpoint(
    elements=elements,
    base_filename=workflow_helpers.WORKFLOW_STAGES["evaluation_completed"],
    metadata=metadata,
)

print("Checkpoint saved to:", checkpoint_path)
print("Metadata saved:")
print(metadata)

Checkpoint saved to: checkpoints\evaluation_completed_20260411_1604.json
Metadata saved:
{'stage': 'evaluation_completed', 'docxfilename': 'custom_word_styles_example.docx', 'source_language': 'English', 'target_language': 'Traditional Chinese', 'primary_model_name': 'gemini-3.1-pro-preview', 'evaluation_model_name': 'gpt-5.4-mini', 'fallback_model_name': None, 'primary_system_message': 'You are a professional translator of Biblical education materials for a Protestant/Evangelical audience.\n\nYou will receive a single JSON object with the following structure:\n\n{\n  "elements": [\n    {\n      "id": "<string>",\n      "text": "<chunked Markdown in English>"\n    },\n    ...\n  ]\n}\n\nEach `text` field is a small chunk of Markdown in English. For each element, you must translate the natural-language prose into Traditional Chinese while preserving formatting and structure.\n\nYou must respond with a single valid JSON object of the form:\n\n{\n  "elements": [\n    {\n      "id": "<same

In [18]:
# Reality check: inspect saved evaluation checkpoint metadata and one example element

import json
from pprint import pprint

with open(checkpoint_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Top-level type:", type(data).__name__)
print("Keys:", list(data.keys()) if isinstance(data, dict) else None)

print("\nMetadata:")
pprint(data.get("metadata") if isinstance(data, dict) else None)

elements_saved = data.get("elements", []) if isinstance(data, dict) else []
print(f"\nSaved element count: {len(elements_saved)}")

print("\nExample saved element:")
pprint(elements_saved[0] if elements_saved else None)

print("\nEvaluation system message preview:")
print((data.get("metadata", {}).get("evaluation_system_message") or "")[:200])

Top-level type: dict
Keys: ['metadata', 'elements']

Metadata:
{'docxfilename': 'custom_word_styles_example.docx',
 'evaluation_model_name': 'gpt-5.4-mini',
 'evaluation_system_message': 'You are a translation validator for Biblical '
                              'education materials intended for a '
                              'Protestant/Evangelical audience.\n'
                              '\n'
                              'Your task is to evaluate whether each candidate '
                              'translation faithfully preserves the meaning of '
                              'the source text and preserves required Markdown '
                              '/ inline formatting.\n'
                              '\n'
                              'Be careful but not overly strict. Allow natural '
                              'translation variation. Do not fail a '
                              'translation simply because it is not literal.\n'
                              '

## Notebook checkpoint and handoff

This notebook completed the evaluation stage of the Word-to-LLM workflow by performing the following steps:

- loaded the prior checkpoint containing metadata and primary translation results
- pulled `source_language`, `target_language`, and `primary_model_name` forward from checkpoint metadata
- defined the evaluator model for this stage
- defined the evaluator system prompt directly in the notebook
- saved the evaluator system prompt into checkpoint metadata for provenance
- identified all elements eligible for evaluation
- reused the existing `batch_number` values for batched evaluator requests
- ran a one-batch smoke test to verify evaluator payload, response parsing, and schema writeback
- executed the full evaluation pass across all eligible batches
- wrote evaluator results into the schema fields
- updated workflow metadata to reflect the completed evaluation stage
- saved the resulting intermediate state to a timestamped JSON checkpoint together with workflow metadata

### Output of this notebook
The main output is a timestamped JSON checkpoint containing:

- a top-level `metadata` block
- an `elements` list containing primary translation results together with evaluator results

This file is intended to be used as input for the next notebook.

### Metadata saved at this stage
The checkpoint metadata currently records:

- `stage = evaluation_completed`
- `docxfilename`
- `source_language`
- `target_language`
- `primary_model_name`
- `evaluation_model_name`
- `primary_system_message`
- `evaluation_system_message`
- placeholder fields for `fallback_model_name` and `fallback_system_message`, which remain `None` at this stage

### Scope of this notebook
This notebook focuses only on evaluating the primary translation output.

At this stage:

- `evaluator_ran` records whether evaluation completed successfully for each eligible element
- `evaluator_passed` records the evaluator decision
- `evaluator_feedback` stores brief failure feedback when applicable
- `evaluator_error` stores any evaluator-stage errors
- this notebook does not produce fallback translations
- this notebook does not assign `final` or `final_model`

### Next step
If any elements failed evaluation, the next notebook will route those failed items to the fallback translator.

If all elements passed evaluation, the fallback stage may be unnecessary for this document, but the next notebook can still be used for testing or for larger documents that produce failures.

Go to: `5_fallback_translation.ipynb`